# 04. 대화형 Agent 완성

---
## 0. 설치 & 경로

In [1]:
!uv pip install duckduckgo-serach

Using Python 3.13.13 environment at: D:\autornd\SK Autonomous R&D\AutoRnDEnv\.venv
  × No solution found when resolving dependencies:
  ╰─▶ Because duckduckgo-serach was not found in the package registry and
      you require duckduckgo-serach, we can conclude that your requirements
      are unsatisfiable.


In [2]:
!uv add openai python-dotenv langchain langchain-openai langchain-core langchain-community langchain-text-splitters langchain-classic faiss-cpu pypdf pytz yfinance requests ddgs tabulate rank_bm25

Resolved 396 packages in 2ms
Checked 360 packages in 45ms


In [3]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

RAG_SAMPLES = os.path.join(os.getcwd(), 'samples', 'pdf_samples')
print('RAG_SAMPLES:', RAG_SAMPLES)
for p in sorted(Path(RAG_SAMPLES).glob('*.pdf')):
    print(' -', p.name)

RAG_SAMPLES: d:\autornd\SK Autonomous R&D\실습\14일차_실습\samples\pdf_samples
 - A survey on large language model based autonomous agents.pdf
 - data2vec.pdf
 - NeurIPS-2023-analyzing-vision-transformers-for-image-classification-in-class-embedding-space-Paper-Conference.pdf
 - Physics-Guided Deep Learning.pdf
 - supervised-contrastive-learning-Paper.pdf
 - 보도자료.pdf
 - 학칙.pdf


---
## Step 1. RAG VectorStore 

In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
import os

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

# 학칙.pdf
loader = PyPDFLoader(os.path.join(RAG_SAMPLES, '학칙.pdf'))
all_splits = text_splitter.split_documents(loader.load())

# news.pdf
loader_news = PyPDFLoader(os.path.join(RAG_SAMPLES, '보도자료.pdf'))
news_splits = text_splitter.split_documents(loader_news.load())
all_splits.extend(news_splits)

embedding = OpenAIEmbeddings(
    model='text-embedding-3-large', api_key=OPENAI_API_KEY
)

persist_directory = 'chroma_store'	

# 저장된 크로마 DB가 없다면 새로 만들기
if not os.path.exists(persist_directory):
    print("Creating new Chroma store")
    vectorstore = Chroma.from_documents(
        documents=all_splits,
        embedding=embedding,
        persist_directory=persist_directory
    )

else:
    print("Loading existing Chroma store")
    vectorstore = Chroma(		
        persist_directory=persist_directory, 
        embedding_function=embedding
    )
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

C:\Users\Admin\AppData\Local\Temp\ipykernel_38180\1014595930.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
d:\autornd\SK Autonomous R&D\AutoRnDEnv\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading existing Chroma store


In [5]:
retriever # 생성한 VectorStore 기반으로 검색을 수행하는 Retriever 객체

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001934E486F90>, search_kwargs={'k': 3})

---
## Step 1.1 검색 품질 점검

`석사학위과정 수업연한`처럼 **별표(부록)에 같은 단어가 반복**되는 PDF에서는, 기본 유사도 검색이 엉뚱한 청크 3개만 가져올 수 있습니다.  
아래에서 검색 결과와 **실제 정답 조항**을 비교해 봅니다.

In [6]:
demo_query = '석사학위과정 수업연한'

retriever_baseline = vectorstore.as_retriever(search_kwargs={'k': 3})
baseline_hits = retriever_baseline.invoke(demo_query)

print(f'질문: {demo_query}\n')
for i, doc in enumerate(baseline_hits, 1):
    has_answer = '수업연한' in doc.page_content and '2년(4학기)' in doc.page_content
    print(f'[{i}] 수업연한 정답 포함: {has_answer}')
    print(doc.page_content[:350].replace('\n', ' '))
    print('-' * 60)

질문: 석사학위과정 수업연한



In [7]:
# 학칙.pdf 본문에 실제로 있는 조항 (검색이 찾아야 할 정답)
answer_chunks = [
    s for s in all_splits
    if '수업연한' in s.page_content and '2년(4학기)' in s.page_content
]

print(f'정답 청크 수: {len(answer_chunks)}')
print('=' * 60)
print(answer_chunks[0].page_content[:700])

정답 청크 수: 1
③ <삭제> 제2조의7(수업연한) ① 대학원 각 학위과정의 수업연한은 다음 각호와 같다.1. 석사학위과정: 2년(4학기)2. 박사학위과정: 2년(4학기)3. 통합과정: 3년(6학기)② 제1항의 규정에도 불구하고 학위수여 자격 요건을 충족한 학생은 수업연한을 1학기 단축할 수 있다. 단, 통합과정 및 석사학위논문 대체실적을 통해 학위를 취득하고자 하는 학생은 제외한다.제2조의8(재학연한) ① 대학원 각 학위과정의 재학연한은 다음 각호와 같으며, 이를 초과할 수 없다. 단, 휴학 및 제적 기간은 재학연한에 산입하지 아니한다. 1. 석사학위과정: 4년(8학기)2. 박사학위과정: 7년(14학기)3. 통합과정: 8년(16학기)② 「장애인 등에 대한 특수교육법」 제15조 제1항의 특수교육대상자로서 절차에 따라 학적에 구분표시 되어 있는 학생에 대하여는 재학연한을 제한하지 아니한다.제2조의9(융합전공) ① 대학원 내 두 개 이상의 학과가 융합적 교과과정(융합전공)을 공동 운영할 수 있다.② 융합전공의 운영에 관한 사항은 별도로 정한다.제2조의10(학생설계전공) ① 새로운 학문수요의 충족과 자기 주도적 학습능력의 향상을 위하여 학생설계전공을 둘 수 있다.② 학생설계전공의 설치 및 이수요건은 별도로 정한다.제3조(입학정원) ① 대학원에 두는 학위과정별 입학정원([별표 1])은 대학원운영위원회에서 대학별로 정한다. 단, 통합과정 입학정원은 박사학위과정


---
## Step 1.2 BM25 키워드 검색 + 하이브리드

**BM25**는 단어(키워드) 빈도 기반 검색입니다. `수업연한`처럼 **문서에 정확히 등장하는 용어**를 찾을 때 벡터 검색보다 유리한 경우가 많습니다.

| 방식 | 장점 | 약점 |
|------|------|------|
| 벡터 검색 | 의미가 비슷한 문장도 찾음 | 별표처럼 같은 단어가 반복되면 엉뚱한 청크가 올라옴 |
| BM25 | 키워드가 들어 있는 청크를 직접 찾음 | 한글 PDF는 띄어쓰기가 적어 **토크나이저 설정**이 필요 |

실습에서는 **BM25 + 벡터**를 `EnsembleRetriever`로 합쳐 하이브리드 검색을 씁니다.

In [8]:
import re
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.tools import create_retriever_tool


def ko_preprocess(text):
    """한글 학칙 PDF용 — 띄어쓰기 대신 한글 어절 단위로 토큰 분리"""
    return re.findall(r'[가-힣]+|\d+', text)


def show_hits(label, hits, preview=3):
    print(f'\n=== {label} ===')
    for i, doc in enumerate(hits[:preview], 1):
        has_answer = '수업연한' in doc.page_content and '2년(4학기)' in doc.page_content
        print(f'[{i}] 수업연한 정답 포함: {has_answer}')
        print(doc.page_content[:280].replace('\n', ' '))
        print('-' * 50)

In [9]:
# BM25 키워드 검색
bm25_retriever = BM25Retriever.from_documents(all_splits, preprocess_func=ko_preprocess)
bm25_retriever.k = 3
bm25_hits = bm25_retriever.invoke(demo_query)
show_hits('BM25 키워드 검색 (k=3)', bm25_hits)


=== BM25 키워드 검색 (k=3) ===
[1] 수업연한 정답 포함: True
③ <삭제> 제2조의7(수업연한) ① 대학원 각 학위과정의 수업연한은 다음 각호와 같다.1. 석사학위과정: 2년(4학기)2. 박사학위과정: 2년(4학기)3. 통합과정: 3년(6학기)② 제1항의 규정에도 불구하고 학위수여 자격 요건을 충족한 학생은 수업연한을 1학기 단축할 수 있다. 단, 통합과정 및 석사학위논문 대체실적을 통해 학위를 취득하고자 하는 학생은 제외한다.제2조의8(재학연한) ① 대학원 각 학위과정의 재학연한은 다음 각호와 같으며, 이를 초과할 수 없다. 단, 휴학 및 제적 기간은 재학
--------------------------------------------------
[2] 수업연한 정답 포함: False
34 [별표 2] 학위과정별, 학과 및 전공별 학위수여명1. 석사학위과정(신촌캠퍼스)대학학과국문학위명영문학위명후드코드 문과대학 국어국문학과문학석사M.A. in Korean Language and LiteraturePANTONE 5315C중어중문학과문학석사M.A. in Chinese Language and LiteraturePANTONE 5315C영어영문학과문학석사M.A. in English Language and LiteraturePANTONE 5315C독어독문학과문학석사M.A. in German
--------------------------------------------------
[3] 수업연한 정답 포함: False
10 제7조의2(편입학) ① 당해 학년도 입학정원에서 재학생 수를 뺀 범위 안에서 편입학생을 모집할 수 있다.② 편입학에 관한 내규는 따로 정한다.제7조의3(재입학) 제적 또는 과정수료 처리된 자는 입학정원의 결원이 있을 때 1회에 한하여 재입학을 신청할 수 있으며, 이에 관한 사항은 따로 정한다.1. <삭제>2. <삭제>3. <삭제>제4장 등록 및 학적 변동제8조(등록) ① 대학원 각 학위과정의 학생은 재학연한 기간 내에 다음 각 

In [10]:
# 하이브리드: BM25 + 벡터
vector_retriever = vectorstore.as_retriever(search_kwargs={'k': 3})
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5],
)
hybrid_hits = hybrid_retriever.invoke(demo_query)
show_hits('하이브리드 BM25 + 벡터 (k=3)', hybrid_hits)

found = any(
    '수업연한' in d.page_content and '2년(4학기)' in d.page_content
    for d in hybrid_hits
)
print('\n수업연한 정답 청크 포함 여부:', found)


=== 하이브리드 BM25 + 벡터 (k=3) ===
[1] 수업연한 정답 포함: True
③ <삭제> 제2조의7(수업연한) ① 대학원 각 학위과정의 수업연한은 다음 각호와 같다.1. 석사학위과정: 2년(4학기)2. 박사학위과정: 2년(4학기)3. 통합과정: 3년(6학기)② 제1항의 규정에도 불구하고 학위수여 자격 요건을 충족한 학생은 수업연한을 1학기 단축할 수 있다. 단, 통합과정 및 석사학위논문 대체실적을 통해 학위를 취득하고자 하는 학생은 제외한다.제2조의8(재학연한) ① 대학원 각 학위과정의 재학연한은 다음 각호와 같으며, 이를 초과할 수 없다. 단, 휴학 및 제적 기간은 재학
--------------------------------------------------
[2] 수업연한 정답 포함: False
34 [별표 2] 학위과정별, 학과 및 전공별 학위수여명1. 석사학위과정(신촌캠퍼스)대학학과국문학위명영문학위명후드코드 문과대학 국어국문학과문학석사M.A. in Korean Language and LiteraturePANTONE 5315C중어중문학과문학석사M.A. in Chinese Language and LiteraturePANTONE 5315C영어영문학과문학석사M.A. in English Language and LiteraturePANTONE 5315C독어독문학과문학석사M.A. in German
--------------------------------------------------
[3] 수업연한 정답 포함: False
10 제7조의2(편입학) ① 당해 학년도 입학정원에서 재학생 수를 뺀 범위 안에서 편입학생을 모집할 수 있다.② 편입학에 관한 내규는 따로 정한다.제7조의3(재입학) 제적 또는 과정수료 처리된 자는 입학정원의 결원이 있을 때 1회에 한하여 재입학을 신청할 수 있으며, 이에 관한 사항은 따로 정한다.1. <삭제>2. <삭제>3. <삭제>제4장 등록 및 학적 변동제8조(등록) ① 대학원 각 학위과정의 학생은 재학연한 기간 내에 다

---
## Step 2. Retriever → Tool (5.3 마지막 섹션)

`langchain_core.tools.create_retriever_tool` — 5.3과 동일합니다.

> **Step 1.2**에서 하이브리드 retriever로 `search_pdf_tool`을 이미 만들었다면, 이 셀은 생략해도 됩니다.

In [11]:
from langchain_core.tools import create_retriever_tool

search_pdf_tool = create_retriever_tool(
    retriever,
    name='search_pdf_documents',
    description='pdf samples 안의 문서함에서 질문과 관련된 내용을 검색합니다.',
)

---
## Step 3. 유틸 Tool (5.2 복습)

시각 · 주가 — `@tool` 로 등록합니다.

In [12]:
from datetime import datetime
import pytz
import yfinance as yf
from langchain_core.tools import tool

@tool
def get_current_time(timezone: str, location: str) -> str:
    """현재 시각. timezone 예: Asia/Seoul, location 예: 서울"""
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime('%Y-%m-%d %H:%M:%S')
    return f'{timezone} ({location}) 현재시각 {now}'

@tool
def get_yf_stock_history(ticker: str, period: str) -> str:
    """주식 가격 조회. ticker 예: TSLA, period 예: 1mo"""
    history = yf.Ticker(ticker).history(period=period)
    return history.tail(3).to_string() if not history.empty else '데이터 없음'


In [13]:
print(get_current_time.invoke({'timezone': 'Asia/Seoul', 'location': '서울'}))

Asia/Seoul (서울) 현재시각 2026-07-09 14:58:32


---
## Step 4. DuckDuckGo 웹 검색

최신 뉴스·시사는 PDF RAG가 아니라 **웹 검색** Tool을 씁니다.  
Agent가 질문마다 `search_pdf_documents` vs `duckduckgo_results_json` 중 선택합니다.

In [14]:
from langchain_community.tools import DuckDuckGoSearchResults

web_search = DuckDuckGoSearchResults(num_results=3)
print(web_search.name, '-', web_search.description[:80], '...')

duckduckgo_results_json - A wrapper around Duck Duck Go Search. Useful for when you need to answer questio ...


In [15]:
web_search.invoke("26년 월드컵 한국 vs 멕시코 점수는?")

'snippet: 분류 홍명보호 (성인 2기)/2026 FIFA 월드컵 멕시코 축구 국가대표팀 경기 한국-멕시코 관계 2026년 축구 경기, title: 홍명보호(성인 2기)/2026 Fifa 월드컵/멕시코전 - 나무위키, link: https://namu.wiki/w/홍명보호(성인+2기)/2026+FIFA+월드컵/멕시코전, snippet: 이번엔 2026 월드컵 2차전 한국 vs 멕시코에 대한 모든 정보와 관전 포인트를 정리 하겠습니다. 체코를 2-1로 뒤집은 역전승의 여운이 채 가시기도 전에 이제 진짜 고비가 다가왔습니다. 6월 19일 오전 10시, 같은 장소인 멕시코 과달라하라에서 이번엔 개최국 멕시코를 상대합니다. 체코전이 "반드시 ..., title: 2026 월드컵 한국 vs 멕시코 완벽 분석 (시간, 피파 랭킹 순위, 역대 전적, 중계) : 네이버 블로그, link: https://blog.naver.com/nyria99/224316761733, snippet: 2026 년 월드컵 경기 한국-멕시코.... 최종 점수는 0-1 입니다 주요 내용 의 텍스트 번역 은 아래 에 있다. -글쎄,내가 무슨 말을 할 수 있니? 물론 멕시코는 강한 국가 대표팀이지만 한국인에게는 가장 좋은 경기는 아닙니다. 한국 골키퍼의 유일한 실수가 아니었다면..., title: 2026 년 월드컵 경기 한국-멕시코.... 최종 점수는 0-1 입니다 주요 내용 의 텍스트 번역 은 아래 에 있다. -글쎄 ..., link: https://vk.com/wall-178306402_6950'

---
## Step 5. AgentExecutor — Tool 통합

5.2 수동 tool 루프 대신 **AgentExecutor**가 tool_calls 루프를 자동 처리합니다.

In [16]:
from langchain_openai import ChatOpenAI
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = ChatOpenAI(model='gpt-5.4-nano', temperature=0.1)

agent_tools = [get_current_time, get_yf_stock_history, search_pdf_tool, web_search]
print([t.name for t in agent_tools])

prompt = ChatPromptTemplate.from_messages([
    ('system', '''너는 사용자를 돕는 AI입니다. 질문에 맞는 도구를 골라 사용하세요.

- samples/rag_samples PDF (학칙, news 등) → search_pdf_documents
- 최신 뉴스·시사·웹 정보 → duckduckgo_results_json
- 주가 → get_yf_stock_history
- 현재 시각 → get_current_time

도구 결과에 없으면 추측하지 마세요. 한국어로 답하세요.'''),
    MessagesPlaceholder('chat_history', optional=True),
    ('human', '{input}'),
    MessagesPlaceholder('agent_scratchpad'),
])

agent = create_tool_calling_agent(llm, agent_tools, prompt)
executor = AgentExecutor(agent=agent, tools=agent_tools, verbose=True, max_iterations=8)

['get_current_time', 'get_yf_stock_history', 'search_pdf_documents', 'duckduckgo_results_json']


In [17]:
# PDF → search_pdf_documents
r = executor.invoke({'input': '석사학위과정 수업연한은?', 'chat_history': []})
print('RAG:', r['output'])



> Entering new AgentExecutor chain...

Invoking: `search_pdf_documents` with `{'query': '석사학위과정 수업연한'}`


제가 가지고 있는 PDF 자료들(samples/rag_samples)에서 **“석사학위과정 수업연한”** 문구를 확인하지 못했습니다.  

어느 학교(또는 규정/학칙) 기준인지 알려주시면, 해당 자료에서 다시 찾아서 정확한 기간을 답해드릴게요. (예: ○○대학교 학칙, 대학원 학사규정 등)

> Finished chain.
RAG: 제가 가지고 있는 PDF 자료들(samples/rag_samples)에서 **“석사학위과정 수업연한”** 문구를 확인하지 못했습니다.  

어느 학교(또는 규정/학칙) 기준인지 알려주시면, 해당 자료에서 다시 찾아서 정확한 기간을 답해드릴게요. (예: ○○대학교 학칙, 대학원 학사규정 등)


In [18]:
search_pdf_tool = create_retriever_tool(
    hybrid_retriever,
    name='search_pdf_documents',
    description='pdf samples 안의 문서함에서 질문과 관련된 내용을 검색합니다.',
)
# PDF → search_pdf_documents

agent_tools = [get_current_time, get_yf_stock_history, search_pdf_tool, web_search]
agent = create_tool_calling_agent(llm, agent_tools, prompt)
executor = AgentExecutor(agent=agent, tools=agent_tools, verbose=True, max_iterations=8)

r = executor.invoke({'input': '석사학위과정 수업연한은?', 'chat_history': []})
print('RAG:', r['output'])



> Entering new AgentExecutor chain...

Invoking: `search_pdf_documents` with `{'query': '석사학위과정 수업연한'}`


③ <삭제> 제2조의7(수업연한) ① 대학원 각 학위과정의 수업연한은 다음 각호와 같다.1. 석사학위과정: 2년(4학기)2. 박사학위과정: 2년(4학기)3. 통합과정: 3년(6학기)② 제1항의 규정에도 불구하고 학위수여 자격 요건을 충족한 학생은 수업연한을 1학기 단축할 수 있다. 단, 통합과정 및 석사학위논문 대체실적을 통해 학위를 취득하고자 하는 학생은 제외한다.제2조의8(재학연한) ① 대학원 각 학위과정의 재학연한은 다음 각호와 같으며, 이를 초과할 수 없다. 단, 휴학 및 제적 기간은 재학연한에 산입하지 아니한다. 1. 석사학위과정: 4년(8학기)2. 박사학위과정: 7년(14학기)3. 통합과정: 8년(16학기)② 「장애인 등에 대한 특수교육법」 제15조 제1항의 특수교육대상자로서 절차에 따라 학적에 구분표시 되어 있는 학생에 대하여는 재학연한을 제한하지 아니한다.제2조의9(융합전공) ① 대학원 내 두 개 이상의 학과가 융합적 교과과정(융합전공)을 공동 운영할 수 있다.② 융합전공의 운영에 관한 사항은 별도로 정한다.제2조의10(학생설계전공) ① 새로운 학문수요의 충족과 자기 주도적 학습능력의 향상을 위하여 학생설계전공을 둘 수 있다.② 학생설계전공의 설치 및 이수요건은 별도로 정한다.제3조(입학정원) ① 대학원에 두는 학위과정별 입학정원([별표 1])은 대학원운영위원회에서 대학별로 정한다. 단, 통합과정 입학정원은 박사학위과정의 입학정원에 산정된다.② 다음 각호의 어느 하나에 해당하는 학생의 정원은 제1항의 규정에 의한 정원 외에 따로 있는 것으로 본다.1. 대학원에 입학·편입학 또는 재입학하는 다음 각 목의 학생가. 교육부령으로 정하는 위탁학생나. 북한이탈주민다. 부모가 모두 외국인인 외국인라. 외국에서 우리나라 초·중등교육과 대학교육에 상응하는 교육과정을 전부

In [19]:
# 시각 → get_current_time
r = executor.invoke({'input': '부산 지금 몇시야?', 'chat_history': []})
print('시각:', r['output'])



> Entering new AgentExecutor chain...

Invoking: `get_current_time` with `{'timezone': 'Asia/Seoul', 'location': '부산'}`


Asia/Seoul (부산) 현재시각 2026-07-09 14:58:45부산 현재 시각은 **2026-07-09 14:58:45** 입니다.

> Finished chain.
시각: 부산 현재 시각은 **2026-07-09 14:58:45** 입니다.


---
## Step 6. 멀티턴 — `chat_history` (5.1 + Agent)

In [20]:
print(executor.invoke({'input': '학칙에서 석사 수업연한 알려줘', 'chat_history': []})['output'])
print(executor.invoke({'input': '방금 말한 내용 한 줄로 다시', 'chat_history': []})['output'])



> Entering new AgentExecutor chain...

Invoking: `search_pdf_documents` with `{'query': '학칙 석사 수업연한'}`


Ⅰ. 대학원 학칙  /  29
[별표 1]학위과정별(캠퍼스) 입학정원 현황(2026학년도)□ 일반 학위과정캠퍼스석사 정원(명)박사 정원(명)신촌1,3281)1,0292)국제12271합계1,4501,1001) 통계데이터사이언스학과(첨단분야 교원확보율 충족형) 석사 정원 8명 증원2) 통계데이터사이언스학과(첨단분야 교원확보율 충족형) 박사 정원 3명 증원□ 계약학과 학위과정 [정원 외]1. 신촌캠퍼스구분석사 정원(명)박사 정원(명)재교육형10028채용조건형11733합계21761

◯148 (시행일) 이 개정 학칙 [별표 1-1] [별표 2]는 2025학년도 입학생부터 적용한다.부칙 <2025. 6. 4.>◯149 (시행일) (1) 이 개정 학칙(제31조 제1항, 제51조)은 2025년 3월 1일부터 시행한 것으로 본다. (2) 이 개정 학칙(제2조의2, 제7조의3, 제13조, [별표 1], [별표 1-1], [별표 2])은 2025년 6월 4일부터 시행한다. (3) 이 개정 학칙 ([별표 1-2] 중 한국전력공사)은 2025년 6월 4일부터 시행한다. (4) 이 개정 학칙 ([별표 1-1], [별표 2] 중 지능형디스플레이공학과)은 2025년 9월 1일부터 시행한다. (5) 이 개정 학칙(제15조)은 2026학년도 1학기 입학생부터 적용한다.부칙 <2025. 08. 22.>◯150 (시행일) (1) 이 개정 학칙 (제5조 제3항)은 2024년 3월 12일부터 시행한 것으로 본다. (2) 이 개정 학칙([별표 2]의 통합디자인학과 석사영문학위명)은 2025년 8월 졸업생부터 적용한다.(3) 이 개정 학칙(제17조 제1항 및 제2항)은 2025년 8월 22일부터 시행한다. (4) 이 개정 학칙([별표 1-1]의 행정학과 석사 영문 학위명은 2025학년도 2학기부터 시행한다.(5)

In [21]:
from langchain_core.messages import HumanMessage, AIMessage

class ChatSession:
    def __init__(self, executor):
        self.executor = executor
        self.history = []

    def ask(self, question: str) -> str:
        result = self.executor.invoke({'input': question, 'chat_history': self.history})
        answer = result['output']
        self.history.extend([HumanMessage(content=question), AIMessage(content=answer)])
        return answer

session = ChatSession(executor)
print('1:', session.ask('학칙에서 석사 수업연한 알려줘'))
print('2:', session.ask('방금 답한 내용 한 줄로 다시'))



> Entering new AgentExecutor chain...

Invoking: `search_pdf_documents` with `{'query': '학칙 석사 수업연한'}`


Ⅰ. 대학원 학칙  /  29
[별표 1]학위과정별(캠퍼스) 입학정원 현황(2026학년도)□ 일반 학위과정캠퍼스석사 정원(명)박사 정원(명)신촌1,3281)1,0292)국제12271합계1,4501,1001) 통계데이터사이언스학과(첨단분야 교원확보율 충족형) 석사 정원 8명 증원2) 통계데이터사이언스학과(첨단분야 교원확보율 충족형) 박사 정원 3명 증원□ 계약학과 학위과정 [정원 외]1. 신촌캠퍼스구분석사 정원(명)박사 정원(명)재교육형10028채용조건형11733합계21761

◯148 (시행일) 이 개정 학칙 [별표 1-1] [별표 2]는 2025학년도 입학생부터 적용한다.부칙 <2025. 6. 4.>◯149 (시행일) (1) 이 개정 학칙(제31조 제1항, 제51조)은 2025년 3월 1일부터 시행한 것으로 본다. (2) 이 개정 학칙(제2조의2, 제7조의3, 제13조, [별표 1], [별표 1-1], [별표 2])은 2025년 6월 4일부터 시행한다. (3) 이 개정 학칙 ([별표 1-2] 중 한국전력공사)은 2025년 6월 4일부터 시행한다. (4) 이 개정 학칙 ([별표 1-1], [별표 2] 중 지능형디스플레이공학과)은 2025년 9월 1일부터 시행한다. (5) 이 개정 학칙(제15조)은 2026학년도 1학기 입학생부터 적용한다.부칙 <2025. 08. 22.>◯150 (시행일) (1) 이 개정 학칙 (제5조 제3항)은 2024년 3월 12일부터 시행한 것으로 본다. (2) 이 개정 학칙([별표 2]의 통합디자인학과 석사영문학위명)은 2025년 8월 졸업생부터 적용한다.(3) 이 개정 학칙(제17조 제1항 및 제2항)은 2025년 8월 22일부터 시행한다. (4) 이 개정 학칙([별표 1-1]의 행정학과 석사 영문 학위명은 2025학년도 2학기부터 시행한다.(5)

---
## Step 7. `input()` 대화 루프

In [22]:
"""
print('종료: exit')
while True:
    user_input = input('사용자: ').strip()
    if user_input.lower() in ('exit', 'quit', '종료'):
        break
    if not user_input:
        continue
    print('AI:', session.ask(user_input))
"""

"\nprint('종료: exit')\nwhile True:\n    user_input = input('사용자: ').strip()\n    if user_input.lower() in ('exit', 'quit', '종료'):\n        break\n    if not user_input:\n        continue\n    print('AI:', session.ask(user_input))\n"

---
## Step 8. 나만의 Tool 만들기

`@tool`로 함수를 등록하고 `agent_tools`에 넣으면 Agent가 사용할 수 있습니다.

1. 함수 정의 (`@tool` + docstring)
2. `.invoke()`로 단독 테스트
3. `agent_tools`에 추가 → prompt 보강 → `executor` 재생성
4. Agent 질문으로 Tool 호출 확인

### 8.1 예시 — `celsius_to_fahrenheit`

섭씨 → 화씨 변환 Tool을 만들고 Agent에 연결합니다.

In [23]:
from langchain_core.tools import tool

@tool
def celsius_to_fahrenheit(celsius: float) -> str:
    """섭씨 온도를 화씨로 변환합니다."""
    fahrenheit = celsius * 9 / 5 + 32
    return f'{celsius}°C = {fahrenheit:.1f}°F'

# 1) Tool 단독 실행
print(celsius_to_fahrenheit.name)
print(celsius_to_fahrenheit.invoke({'celsius': 25}))

celsius_to_fahrenheit
25.0°C = 77.0°F


In [24]:
# 2) agent_tools에 추가 후 Agent 재생성
agent_tools = agent_tools + [celsius_to_fahrenheit]

prompt = ChatPromptTemplate.from_messages([
    ('system', '''너는 사용자를 돕는 AI입니다. 질문에 맞는 도구를 골라 사용하세요.

- samples/rag_samples PDF (학칙, news 등) → search_pdf_documents
- 최신 뉴스·시사·웹 정보 → duckduckgo_results_json
- 주가 → get_yf_stock_history
- 현재 시각 → get_current_time
- 섭씨→화씨 온도 변환 → celsius_to_fahrenheit

도구 결과에 없으면 추측하지 마세요. 한국어로 답하세요.'''),
    MessagesPlaceholder('chat_history', optional=True),
    ('human', '{input}'),
    MessagesPlaceholder('agent_scratchpad'),
])

agent = create_tool_calling_agent(llm, agent_tools, prompt)
executor = AgentExecutor(agent=agent, tools=agent_tools, verbose=True, max_iterations=8)

print('등록된 Tool:', [t.name for t in agent_tools])

등록된 Tool: ['get_current_time', 'get_yf_stock_history', 'search_pdf_documents', 'duckduckgo_results_json', 'celsius_to_fahrenheit']


In [25]:
# 3) Agent가 새 Tool을 호출하는지 확인
r = executor.invoke({'input': '섭씨 36.5도는 화씨로 몇 도야?', 'chat_history': []})
print(r['output'])



> Entering new AgentExecutor chain...

Invoking: `celsius_to_fahrenheit` with `{'celsius': 36.5}`


36.5°C = 97.7°F섭씨 36.5도는 **화씨 97.7도(°F)** 입니다.

> Finished chain.
섭씨 36.5도는 **화씨 97.7도(°F)** 입니다.


### 8.2 실습 — 나만의 Tool

원하는 기능의 Tool을 직접 만들고, Agent가 호출하는지 확인합니다.

- 예: 단위 변환, 문자열 처리, 간단한 계산, 로컬 파일 읽기 등
- docstring에 **언제 쓸지**를 적어야 Agent가 선택합니다.

실시간으로 환율을 환전하는 Tool을 만들고, agent에 연결하는 코드를 작성하고, 검증해줘
Step 8.의 예제를 확인하고, 비슷한 방식으로 agent에 연결해줘
함수는 출발 국가이름 또는 화폐 이름 / 도착 국가이름 / 환전 금액 (ex. '미국' 또는 '미국 달러' 'USD' 식으로)과 금액을 input으로 받고, 도착 국가이름과 최종 환전 금액을 return으로 하는 식으로 작성하면 될 것 같아

In [26]:
import re
import requests
from langchain_core.tools import tool


def _get_supported_currencies() -> dict[str, str]:
    """Frankfurter 지원 통화: ISO 코드 → 영문 화폐명."""
    resp = requests.get('https://api.frankfurter.app/currencies', timeout=10)
    resp.raise_for_status()
    return resp.json()


def _resolve_currency(name: str, currencies: dict[str, str]) -> tuple[str | None, str]:
    """국가명/화폐명/코드를 ISO 코드로 해석. (code, display_name) 반환.

    1) 이미 ISO 코드면 그대로 사용
    2) 영문 화폐명 부분 일치
    3) 그 외(한국어 국가명 등)는 LLM이 지원 통화 목록에서 매칭
    """
    key = name.strip()
    if not key:
        return None, ''

    upper = key.upper()
    if upper in currencies:
        return upper, currencies[upper]

    key_lower = key.lower()
    for code, eng_name in currencies.items():
        if key_lower == eng_name.lower() or key_lower in eng_name.lower():
            return code, eng_name

    # 사전 정의 없이 모델이 국가·화폐명을 지원 통화에 매칭
    catalog = '\n'.join(f'{code}: {eng}' for code, eng in sorted(currencies.items()))
    msg = (
        '다음 사용자 입력(국가명 또는 화폐명)을 아래 ISO 통화 코드 중 하나로 매핑하세요.\n'
        f'사용자 입력: {key}\n\n'
        f'지원 통화:\n{catalog}\n\n'
        '규칙:\n'
        '- 반드시 위 목록의 3글자 코드만 출력\n'
        '- 매핑 불가하면 NONE\n'
        '- 코드만 출력 (설명·따옴표 금지)'
    )
    raw = llm.invoke(msg).content.strip().upper()
    match = re.search(r'\b([A-Z]{3})\b', raw)
    if match and match.group(1) in currencies:
        code = match.group(1)
        return code, currencies[code]
    return None, ''


@tool
def convert_currency(from_currency: str, to_currency: str, amount: float) -> str:
    """실시간 환율로 금액을 환전합니다.
    출발/도착은 국가명(미국, 한국), 화폐명(달러, 원), 통화코드(USD, KRW) 모두 가능합니다.
    환전·환율·외화 변환 질문에 사용하세요."""
    if amount <= 0:
        return '환전 금액은 0보다 커야 합니다.'

    try:
        currencies = _get_supported_currencies()
    except Exception as e:
        return f'지원 통화 목록 조회 실패: {e}'

    from_code, from_name = _resolve_currency(from_currency, currencies)
    to_code, to_name = _resolve_currency(to_currency, currencies)
    if not from_code:
        return f'알 수 없는 출발 통화/국가: {from_currency}'
    if not to_code:
        return f'알 수 없는 도착 통화/국가: {to_currency}'

    try:
        resp = requests.get(
            'https://api.frankfurter.app/latest',
            params={'from': from_code, 'to': to_code, 'amount': amount},
            timeout=10,
        )
        resp.raise_for_status()
        data = resp.json()
        converted = data['rates'][to_code]
        return (
            f'도착 국가/통화: {to_currency} ({to_code}, {to_name})\n'
            f'최종 환전 금액: {converted:,.2f} {to_code}\n'
            f'(기준일: {data["date"]}, {amount:,.2f} {from_code} → {to_code})'
        )
    except Exception as e:
        return f'환율 조회 실패: {e}'


# 1) Tool 단독 실행 (LLM 매칭 포함 — 상위 셀에서 llm 이 정의되어 있어야 함)
print(convert_currency.name)
print(convert_currency.invoke({'from_currency': '미국', 'to_currency': '한국', 'amount': 100}))
print(convert_currency.invoke({'from_currency': 'USD', 'to_currency': '일본', 'amount': 50}))
print(convert_currency.invoke({'from_currency': '브라질', 'to_currency': '스위스', 'amount': 200}))


convert_currency
도착 국가/통화: 한국 (KRW, South Korean Won)
최종 환전 금액: 150,908.00 KRW
(기준일: 2026-07-08, 100.00 USD → KRW)
도착 국가/통화: 일본 (JPY, Japanese Yen)
최종 환전 금액: 8,124.00 JPY
(기준일: 2026-07-08, 50.00 USD → JPY)
도착 국가/통화: 스위스 (CHF, Swiss Franc)
최종 환전 금액: 31.37 CHF
(기준일: 2026-07-08, 200.00 BRL → CHF)


In [27]:
# 2) agent_tools에 추가 후 Agent 재생성
agent_tools = agent_tools + [convert_currency]

prompt = ChatPromptTemplate.from_messages([
    ('system', '''너는 사용자를 돕는 AI입니다. 질문에 맞는 도구를 골라 사용하세요.

- samples/rag_samples PDF (학칙, news 등) → search_pdf_documents
- 최신 뉴스·시사·웹 정보 → duckduckgo_results_json
- 주가 → get_yf_stock_history
- 현재 시각 → get_current_time
- 섭씨→화씨 온도 변환 → celsius_to_fahrenheit
- 실시간 환율 환전 → convert_currency

도구 결과에 없으면 추측하지 마세요. 한국어로 답하세요.'''),
    MessagesPlaceholder('chat_history', optional=True),
    ('human', '{input}'),
    MessagesPlaceholder('agent_scratchpad'),
])

agent = create_tool_calling_agent(llm, agent_tools, prompt)
executor = AgentExecutor(agent=agent, tools=agent_tools, verbose=True, max_iterations=8)

print('등록된 Tool:', [t.name for t in agent_tools])


등록된 Tool: ['get_current_time', 'get_yf_stock_history', 'search_pdf_documents', 'duckduckgo_results_json', 'celsius_to_fahrenheit', 'convert_currency']


In [32]:
# 3) Agent가 새 Tool을 호출하는지 확인
r = executor.invoke({
    'input': '미국 달러 100달러를 한국 원으로 환전하면 얼마야?',
    'chat_history': [],
})
print(r['output'])

r = executor.invoke({
    'input': '캐나다 달러 3000달러를 한국 원으로 환전하면 얼마야?',
    'chat_history': [],
})
print(r['output'])

r = executor.invoke({
    'input': '듕국돈 200을 한국 원으로 환전하면 얼마야?',
    'chat_history': [],
})
print(r['output'])



> Entering new AgentExecutor chain...

Invoking: `convert_currency` with `{'from_currency': 'USD', 'to_currency': 'KRW', 'amount': 100}`


도착 국가/통화: KRW (KRW, South Korean Won)
최종 환전 금액: 150,908.00 KRW
(기준일: 2026-07-08, 100.00 USD → KRW)미국 달러 **100달러**를 한국 원(**KRW**)으로 환전하면 **약 150,908원**입니다. (기준일: 2026-07-08)

> Finished chain.
미국 달러 **100달러**를 한국 원(**KRW**)으로 환전하면 **약 150,908원**입니다. (기준일: 2026-07-08)


> Entering new AgentExecutor chain...

Invoking: `convert_currency` with `{'from_currency': '캐나다', 'to_currency': '한국', 'amount': 3000}`


도착 국가/통화: 한국 (KRW, South Korean Won)
최종 환전 금액: 3,191,673.00 KRW
(기준일: 2026-07-08, 3,000.00 CAD → KRW)캐나다 달러(CAD) **3,000달러**를 한국 원(KRW)으로 환전하면 **약 3,191,673원**입니다.  
(기준일: 2026-07-08)

> Finished chain.
캐나다 달러(CAD) **3,000달러**를 한국 원(KRW)으로 환전하면 **약 3,191,673원**입니다.  
(기준일: 2026-07-08)


> Entering new AgentExecutor chain...

Invoking: `convert_currency` with `{'from_currency': '중국', 'to_currency': '한국', 'amount': 200}`


도착 국가/통화: 한국 (KRW, South K